# 📖 Notebook 3: Stories & Ephemeral Content

Instagram Stories are photos or videos that **disappear after 24 hours**.  
Over 500 million people use Stories daily — it's one of Instagram's most popular features.

From a system design perspective, Stories are fascinating because they introduce  
**time-based expiration** — data that needs to be automatically cleaned up.

## Learning Objectives

By the end of this notebook, you'll understand:
- How Stories differ from regular posts (data model, lifecycle)
- Using **Redis TTL** (Time To Live) for automatic expiration
- The **story ring** — efficiently querying "which people I follow have active stories?"
- View tracking — knowing who watched your story
- Storage optimization for ephemeral content

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/instagram
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `instagram_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import time
import json
from datetime import datetime, timedelta

# ── Connections ───────────────────────────────────────────
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "instagram_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM stories")
    print(f"✅ PostgreSQL — {cur.fetchone()[0]} stories in database")
    cur.execute("SELECT COUNT(*) FROM stories WHERE expires_at > NOW()")
    print(f"   {cur.fetchone()[0]} active (not expired) stories")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print(f"✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 Stories vs Posts: What's Different?

| Feature | Regular Post | Story |
|---------|-------------|-------|
| **Lifetime** | Permanent | 24 hours |
| **Appears in** | Feed (scrolling) | Story ring (top of app) |
| **Interactions** | Likes, comments | Views, replies (DM) |
| **Storage** | Keep forever | Delete after expiry |
| **Multiple per day** | Unusual | Very common (5-10+) |

The key design challenge: **automatic expiration**.  
We need stories to disappear exactly 24 hours after creation — no manual cleanup.

```
Story lifecycle:

Created ──────────────── 24 hours ──────────────── Expired
  │                                                    │
  ├─ Visible in story ring                             ├─ Gone from story ring
  ├─ Viewers can see it                                ├─ Media can be deleted
  └─ Views are tracked                                 └─ View data archived
```

## ❌ Bad Practice: Cron Job Polling the Database

Before we see the good way, let's look at a naive approach many beginners try:  
**a cron job that polls the database every minute looking for expired stories.**

```python
# BAD — runs every 60 seconds, scans the whole stories table
def cleanup_cron_job():
    cur.execute("DELETE FROM stories WHERE expires_at < NOW()")
    # and delete from S3, and remove from any cache, and ...
```

Why this is bad:

| Problem | Why It Hurts |
|---------|-------------|
| **Polling wastes CPU** | Most runs find nothing to delete — still hits the DB |
| **Expiry is imprecise** | A story created at 12:00:30 might linger until 12:01:00 |
| **Reads must re-check time** | Every `SELECT` needs `WHERE expires_at > NOW()` |
| **Hot table writes** | DELETE on a busy table causes index churn and vacuum pressure |
| **Single point of failure** | If the cron skips a run, old stories leak |

Let's run the bad version once to feel how clunky it is, then switch to Redis TTL.

In [ ]:
# ❌ Bad practice: scan the DB for expired rows
conn = get_db()
cur = conn.cursor()
start = time.time()
cur.execute("SELECT COUNT(*) FROM stories WHERE expires_at < NOW()")
expired = cur.fetchone()[0]
elapsed = (time.time() - start) * 1000
conn.close()

print(f"🐢 Cron-style scan found {expired} expired stories in {elapsed:.1f}ms")
print("   Imagine running this every minute across billions of rows...")
print("   → A better approach: let Redis TTL delete data automatically.")

## ✅ Good Practice: Creating a Story with Redis TTL

When a user posts a Story, we:
1. Save the story metadata to PostgreSQL (with an `expires_at` timestamp — for analytics/audit)
2. Cache the story in Redis with a **TTL** (Time To Live) that matches the 24-hour window
3. Add the user to the "active stories" set in Redis

Redis TTL is perfect here — Redis will **automatically delete** the key after the TTL expires.  
No cron jobs, no cleanup scripts, no manual deletion needed. Expiry is precise to the second.

In [ ]:
STORY_TTL_SECONDS = 24 * 60 * 60  # 24 hours = 86400 seconds

def create_story(author_id: int, media_key: str) -> dict:
    """
    Create a new story that expires in 24 hours.
    - Saves to PostgreSQL (permanent record for analytics)
    - Caches in Redis with TTL (for fast serving during the 24-hour window)
    """
    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    expires_at = datetime.now() + timedelta(seconds=STORY_TTL_SECONDS)

    # Save to PostgreSQL
    cur.execute(
        """INSERT INTO stories (author_id, media_key, expires_at)
           VALUES (%s, %s, %s)
           RETURNING id, created_at""",
        (author_id, media_key, expires_at)
    )
    story_id, created_at = cur.fetchone()
    conn.commit()
    conn.close()

    # Cache in Redis with TTL
    story_data = json.dumps({
        "id": story_id,
        "author_id": author_id,
        "media_key": media_key,
        "created_at": str(created_at),
        "expires_at": str(expires_at)
    })

    # Key: story:{story_id} — auto-expires after 24 hours
    r.setex(f"story:{story_id}", STORY_TTL_SECONDS, story_data)

    # Add story_id to the user's active stories list
    # Key: user_stories:{author_id} — sorted set with creation timestamp as score
    r.zadd(f"user_stories:{author_id}", {str(story_id): time.time()})

    # Mark this user as having active stories
    # Key: active_story_users — set of user IDs with active stories
    r.sadd("active_story_users", str(author_id))

    ttl = r.ttl(f"story:{story_id}")
    print(f"📸 Story #{story_id} created by user {author_id}")
    print(f"   Media: {media_key}")
    print(f"   Expires at: {expires_at}")
    print(f"   Redis TTL: {ttl} seconds ({ttl // 3600}h {(ttl % 3600) // 60}m)")

    return {"story_id": story_id, "author_id": author_id, "ttl": ttl}

# Create a few stories
story1 = create_story(1, "stories/user_1/morning_coffee.jpg")
print()
story2 = create_story(1, "stories/user_1/lunch_selfie.jpg")
print()
story3 = create_story(5, "stories/user_5/sunset_view.jpg")

## ⏰ Redis TTL: Automatic Expiration

Let's see TTL in action. We'll create a short-lived story (30 seconds) and watch it expire.

In [ ]:
# Create a story with a very short TTL for demonstration
r = get_redis()

# Store a demo story that expires in 30 seconds
demo_key = "story:demo_ttl"
r.setex(demo_key, 30, json.dumps({"content": "This story disappears in 30 seconds!"}))

print("Created a demo story with 30-second TTL")
print(f"   → Open RedisInsight (http://localhost:5540) and search for key '{demo_key}'")
print(f"   → Watch the TTL count down!\n")

# Check TTL at different times
for i in range(4):
    ttl = r.ttl(demo_key)
    exists = r.exists(demo_key)
    if exists:
        print(f"   ⏱️  T+{i*10}s: TTL = {ttl}s — story exists ✅")
    else:
        print(f"   ⏱️  T+{i*10}s: TTL = {ttl} — story EXPIRED ❌ (automatically deleted!)")
        break
    if i < 3:
        time.sleep(10)

print("\n💡 Redis deleted the key automatically — no cleanup code needed!")
print("   This is exactly how Instagram Stories expire.")

## 💍 The Story Ring

At the top of the Instagram app, you see a row of circles (the "story ring").  
Each circle represents a user you follow who has active (non-expired) stories.

```
┌──────────────────────────────────────────────────────────┐
│  (You)   (Alice)  (Bob)   (User5)  (User10)  ...        │
│   🔴       🔴      🔴      🔴        🔴                  │
│  Your    Alice's  Bob's   User5's  User10's              │
│  Story   Story    Story   Story    Story                 │
└──────────────────────────────────────────────────────────┘
```

To build this, we need to answer: **"Which users I follow have active stories right now?"**  
This query runs every time you open the app — it needs to be fast.

In [ ]:
def get_story_ring(user_id: int) -> list:
    """
    Get the story ring for a user:
    1. Get list of people this user follows
    2. Check which of them have active stories in Redis
    3. Return those users (sorted: unseen first, then by recency)
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    r = get_redis()
    start = time.time()

    # Step 1: Get followed users
    cur.execute(
        "SELECT followee_id FROM follows WHERE follower_id = %s",
        (user_id,)
    )
    followee_ids = [row["followee_id"] for row in cur.fetchall()]

    # Step 2: Check which followed users have active stories
    # We use the "active_story_users" set for a fast intersection
    active_users = r.smembers("active_story_users")
    users_with_stories = [uid for uid in followee_ids if str(uid) in active_users]

    # Step 3: For each user with stories, get their story count
    story_ring = []
    for uid in users_with_stories:
        story_count = r.zcard(f"user_stories:{uid}")
        if story_count > 0:
            # Check if user has viewed the latest story
            cur.execute(
                "SELECT username, display_name FROM users WHERE id = %s",
                (uid,)
            )
            user_info = cur.fetchone()
            story_ring.append({
                "user_id": uid,
                "username": user_info["username"],
                "display_name": user_info["display_name"],
                "story_count": story_count
            })

    elapsed = (time.time() - start) * 1000
    conn.close()

    print(f"💍 Story ring for user {user_id}:")
    print(f"   Follows {len(followee_ids)} users, {len(story_ring)} have active stories")
    print(f"   Query time: {elapsed:.1f}ms")

    return story_ring

# Get story ring for User 10 (who should follow users 1 and 5)
ring = get_story_ring(user_id=10)
if ring:
    print(f"\n   Story Ring:")
    for entry in ring:
        print(f"   🔴 {entry['display_name']} (@{entry['username']}) — {entry['story_count']} stories")
else:
    print("\n   No stories to show (try running the 'create_story' cells above first)")

## 👁️ View Tracking

Instagram shows you who viewed your story. This requires tracking every view  
without slowing down the story viewing experience.

We use Redis sets for fast, deduplicated view tracking:

In [ ]:
def view_story(story_id: int, viewer_id: int):
    """
    Record that a user viewed a story.
    Uses Redis set for fast deduplication (a user can only view once).
    Also saves to PostgreSQL for permanent analytics.
    """
    r = get_redis()
    conn = get_db()
    cur = conn.cursor()

    # Check if story still exists (not expired)
    story_data = r.get(f"story:{story_id}")
    if not story_data:
        print(f"❌ Story #{story_id} has expired or doesn't exist")
        conn.close()
        return

    # Add to Redis view set (returns 1 if new, 0 if already viewed)
    is_new_view = r.sadd(f"story_views:{story_id}", str(viewer_id))

    # Set TTL on view set to match story expiration
    story_ttl = r.ttl(f"story:{story_id}")
    if story_ttl > 0:
        r.expire(f"story_views:{story_id}", story_ttl)

    if is_new_view:
        # Save to PostgreSQL for permanent records
        cur.execute(
            """INSERT INTO story_views (story_id, viewer_id)
               VALUES (%s, %s)
               ON CONFLICT DO NOTHING""",
            (story_id, viewer_id)
        )
        conn.commit()
        print(f"👁️  User {viewer_id} viewed story #{story_id} (new view)")
    else:
        print(f"👁️  User {viewer_id} already viewed story #{story_id} (duplicate ignored)")

    conn.close()

def get_story_viewers(story_id: int) -> list:
    """Get all viewers of a story (from Redis for speed)."""
    r = get_redis()
    viewers = r.smembers(f"story_views:{story_id}")
    return [int(v) for v in viewers]

# Simulate some views on story1
sid = story1["story_id"]
print(f"Simulating views on story #{sid}:\n")

for viewer_id in [2, 3, 5, 7, 10, 15]:
    view_story(sid, viewer_id)

# Try a duplicate view
print()
view_story(sid, 5)  # User 5 views again — should be deduplicated

print(f"\n📊 Total unique viewers: {len(get_story_viewers(sid))}")
print(f"   Viewer IDs: {sorted(get_story_viewers(sid))}")

## 🔄 Story Expiration & Cleanup

Redis TTL handles removing the cached story data automatically.  
But we also need to:
1. Remove the user from the "active_story_users" set when all their stories expire
2. Clean up the user_stories sorted set
3. Optionally delete the media from S3/MinIO

In production, this is typically handled by a **background cleanup job**.

In [ ]:
def cleanup_expired_stories():
    """
    Background job that cleans up expired story data.
    In production, this would run periodically (e.g., every 5 minutes).
    """
    r = get_redis()
    conn = get_db()
    cur = conn.cursor()
    cleaned = 0

    # Get all users marked as having active stories
    active_users = r.smembers("active_story_users")

    for user_id_str in active_users:
        # Get their story IDs
        story_ids = r.zrange(f"user_stories:{user_id_str}", 0, -1)

        # Check which stories have expired (their Redis key no longer exists)
        expired_ids = []
        for sid in story_ids:
            if not r.exists(f"story:{sid}"):
                expired_ids.append(sid)

        # Remove expired stories from the user's sorted set
        if expired_ids:
            r.zrem(f"user_stories:{user_id_str}", *expired_ids)
            cleaned += len(expired_ids)

        # If user has no more active stories, remove from active set
        remaining = r.zcard(f"user_stories:{user_id_str}")
        if remaining == 0:
            r.srem("active_story_users", user_id_str)
            print(f"   Removed user {user_id_str} from active_story_users (no stories left)")

    conn.close()
    print(f"🧹 Cleanup complete: {cleaned} expired story references removed")

# Run cleanup
cleanup_expired_stories()

## 📊 Story Architecture Summary

```
┌─────────┐    POST /stories    ┌───────────┐
│ Client  │────────────────────►│  Story    │
│         │                     │  Service  │
└─────────┘                     └─────┬─────┘
                                      │
                        ┌─────────────┼─────────────┐
                        │             │             │
                        ▼             ▼             ▼
                  ┌──────────┐  ┌──────────┐  ┌──────────┐
                  │PostgreSQL│  │  Redis   │  │  MinIO   │
                  │          │  │          │  │  (S3)    │
                  │ stories  │  │ story:ID │  │  media   │
                  │ table    │  │ (TTL 24h)│  │  bytes   │
                  │ (perm.)  │  │          │  │          │
                  └──────────┘  │ user_    │  └──────────┘
                                │ stories: │
                                │ {uid}    │
                                │          │
                                │ active_  │
                                │ story_   │
                                │ users    │
                                └──────────┘
```

### Redis Keys Used

| Key Pattern | Type | TTL | Purpose |
|-------------|------|-----|----------|
| `story:{id}` | String (JSON) | 24h | Story data cache |
| `user_stories:{uid}` | Sorted Set | ∞ (cleaned up) | User's active story IDs |
| `story_views:{id}` | Set | Same as story | Viewer IDs for a story |
| `active_story_users` | Set | ∞ (cleaned up) | Users with active stories |

## 🧠 Key Takeaways

1. **Redis TTL** is perfect for ephemeral content — set it once, Redis handles deletion
2. **Dual storage**: PostgreSQL for permanent records/analytics, Redis for fast serving
3. **Story ring** query needs to be fast — use Redis sets to intersect "who I follow" with "who has stories"
4. **View tracking** uses Redis sets for deduplication (SADD returns 0 if already a member)
5. **Background cleanup** handles cascading deletes (user_stories set, active_story_users)

### Interview Tips

- Mention TTL immediately when discussing ephemeral content
- Explain the dual-storage pattern: Redis for serving (fast), DB for analytics (permanent)
- Discuss what happens when Redis goes down (fall back to DB query with `WHERE expires_at > NOW()`)
- Mention storage optimization: delete media from S3 after expiry to save costs

### What's Next?

In **Notebook 4**, we'll build Instagram's **Explore page** — a recommendation system  
that suggests posts from accounts you don't follow based on your engagement patterns.